# Forecasting PINN Benchmark

## 1. Configure Training


In [ ]:
import importlib
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import src.utils.dataloader as dataloader_module
import src.utils.training as training_module
import src.experiments.forecasting_scan as forecasting_module

importlib.reload(dataloader_module)
importlib.reload(training_module)
importlib.reload(forecasting_module)

from src.models.pinn import PseudomonasBIOSODE
from src.utils.training import TrainingConfig

run_forecasting = forecasting_module.run_independent_forecasting_experiment
summarize_forecasting = forecasting_module.summarize_forecasting

SELECTED_REACTORS = (
    "AMBR1_14", "AMBR1_15", "AMBR1_16", "AMBR1_17", "AMBR1_18",
    "AMBR1_21", "AMBR1_22", "AMBR1_23", "AMBR1_24",
    "AMBR2_5", "AMBR2_6", "AMBR2_7", "AMBR2_8", "AMBR2_9",
    "AMBR2_10", "AMBR2_13", "AMBR2_14", "AMBR2_15", "AMBR2_16",
    "AMBR2_19", "AMBR2_20", "AMBR2_21",
)

experiment_ids = SELECTED_REACTORS
observation_fractions = (0.8,0.7,0.6,0.5,0.4,0.3,0.2,0.1)
seeds = (42,)
epochs = 10000
experiment_name = "FCT"

cfg = TrainingConfig(
    processed_csv="data/processed/ambr_preprocessed.csv",
    results_dir="results",
    experiment_ids=experiment_ids,
    experiment_name=experiment_name,
    seed=seeds[0],
    n_neurons=20,
    n_hidden_layers=7,
    num_epochs=epochs,
    learning_rate=1e-4,
    data_loss_weight=1.0,
    residual_loss_weight=1.0,
    auxiliary_loss_weight=1.0,
    regularization_loss_weight=1.0,
    use_auxiliary_loss=True,
    use_regularization_loss=False,
    use_softadapt=True,
    track_epoch_r2=True,
    use_early_stopping=False,
    validation_strategy="none",
    obs_fit_weights=(1.0, 1.0, 0.1, 0.1),
    aux_fit_weights=(1.0,) * len(PseudomonasBIOSODE.state_names),
    res_eq_weights=(1e-15,) * len(PseudomonasBIOSODE.state_names),
    reg_eq_weights=(1e-6,) * len(PseudomonasBIOSODE.state_names),
)

## 2. Run Training


In [ ]:
forecast = run_forecasting(
    cfg,
    observation_fractions=observation_fractions,
    seeds=seeds,
    keep_results=False,
)

run_dirs = forecast["run_dirs"]
run_dirs

## 3. Build Summary

In [ ]:
results_dir = PROJECT_ROOT / "results" / experiment_name
summary = summarize_forecasting(results_dir)
summary_path = summary["table"]
summary_df = pd.read_csv(summary_path)
display(summary_df)